# Perpetrator PCA train-only — deterministic weighted search

Versión rápida/reproducible: busca multiplicador de class weight positivo con threshold fijo 0.50.

In [ ]:

# ==============================
# PERPETRATOR — PCA TRAIN-ONLY — DETERMINISTIC WEIGHT SEARCH
# ==============================
# Objetivo: reproducibilidad razonable sin grid completo.
# Estrategia:
#   - split fijo
#   - scaler/PCA fit SOLO en train
#   - semillas fijadas
#   - sin mixed precision
#   - búsqueda reducida de arquitecturas + multiplicador explícito de class_weight positivo
#   - threshold fijo = 0.50
#   - selección por criterio fijo de screening: recall_1 alto y specificity mínima

# ==============================
# CONFIGURACIÓN
# ==============================
import os
from pathlib import Path

OUTPUT_DIR = './content/perpetrator'   # mantener carpeta pedida

RANDOM_STATE = 42
FORCE_CPU_FOR_REPRODUCIBILITY = False  # si quieres máxima reproducibilidad CPU: True; si tarda mucho: False
ENABLE_STRICT_OP_DETERMINISM = False   # True puede ser MUY lento

BATCH_SIZE = 128
PCA_THRESHOLD = 0.99
THRESHOLD = 0.50
EPOCHS = 80
PATIENCE = 6
LEARNING_RATE = 1e-3

# Criterio objetivo
TARGET_RECALL_POS = 0.90
MIN_RECALL_NEG = 0.20
FALLBACK_RECALL_POS = 0.85

# Multiplicadores de peso positivo. Esto reemplaza la variabilidad aleatoria por una búsqueda explícita y reproducible.
# class_weight[1] = ratio_neg_pos * POS_WEIGHT_MULTIPLIER
POS_WEIGHT_MULTIPLIERS = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]

# Arquitecturas reducidas: incluye las dos candidatas que en el grid anterior llegaban al criterio,
# más variantes cercanas para no depender de una sola.
CANDIDATES = [
    (64, 'linear',  0.3,  4, 'relu',   0.10),  # grid anterior índice ~132
    (64, 'sigmoid', 0.3, 16, 'linear', 0.00),  # grid anterior índice ~329
    (64, 'linear',  0.2,  4, 'relu',   0.10),
    (64, 'linear',  0.3,  4, 'relu',   0.05),
    (64, 'relu',    0.3, 16, 'relu',   0.00),
    (64, 'relu',    0.3, 16, 'linear', 0.05),
]

TRAINING_SEEDS = [42]  # si quieres comprobar robustez, poner [42, 123], pero tardará el doble

# Si se alcanza un candidato target suficientemente bueno, salir antes.
EARLY_EXIT_IF_STRONG_TARGET = True
EARLY_EXIT_MIN_RECALL0 = 0.30

if FORCE_CPU_FOR_REPRODUCIBILITY:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)

# ==============================
# IMPORTS
# ==============================
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import joblib

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, Callback

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
tf.keras.mixed_precision.set_global_policy('float32')

if ENABLE_STRICT_OP_DETERMINISM:
    try:
        tf.config.experimental.enable_op_determinism()
        print('TensorFlow op determinism: ON')
    except Exception as e:
        print('No se pudo activar enable_op_determinism:', repr(e))
else:
    print('TensorFlow op determinism: OFF')

print('TF version:', tf.__version__)
print('GPUs visibles:', tf.config.list_physical_devices('GPU'))
print('Política precisión:', tf.keras.mixed_precision.global_policy())

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Limpiar archivos críticos para evitar leer resultados viejos si algo falla.
for fname in [
    'predictions_with_probs.csv', 'best_report.json', 'gridsearch_results.csv',
    'metrics_final.csv', 'config.json', 'best_model.h5'
]:
    p = Path(OUTPUT_DIR) / fname
    if p.exists():
        p.unlink()

# ==============================
# UTILIDADES
# ==============================
def read_csv_fallback(primary, fallback):
    if os.path.exists(primary):
        return pd.read_csv(primary)
    if os.path.exists(fallback):
        return pd.read_csv(fallback)
    raise FileNotFoundError(f'No encuentro ni {primary!r} ni {fallback!r}')


def pca_variance_df(pca_model):
    explained = pca_model.explained_variance_ratio_
    return pd.DataFrame({
        'PC': [f'PC{i+1}' for i in range(len(explained))],
        'Explained_Variance': explained,
        'Cumulative_Variance': np.cumsum(explained),
    })


def select_n_components(df_var, threshold):
    mask = df_var['Cumulative_Variance'] >= threshold
    if not mask.any():
        return len(df_var)
    return int(mask.idxmax() + 1)


def make_dataset(X, y, batch_size, seed=None, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X.astype('float32'), y.astype('float32')))
    if training:
        ds = ds.shuffle(buffer_size=len(y), seed=seed, reshuffle_each_iteration=False)
    ds = ds.batch(batch_size)
    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)
    return ds


def build_model(input_dim, params, seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    u1, a1, d1, u2, a2, d2 = params
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(u1, activation=a1,
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 1),
                     bias_initializer='zeros'),
        layers.Dropout(d1, seed=seed + 11),
        layers.Dense(u2, activation=a2,
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 2),
                     bias_initializer='zeros'),
        layers.Dropout(d2, seed=seed + 12),
        layers.Dense(1, activation='sigmoid',
                     kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 3),
                     bias_initializer='zeros', dtype='float32'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        ],
    )
    return model


def metric_row(y_true, y_prob, params, seed, epochs_ran, pos_weight_multiplier):
    y_pred = (y_prob >= THRESHOLD).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall0 = report.get('0', {}).get('recall', 0.0)
    recall1 = report.get('1', {}).get('recall', 0.0)
    precision1 = report.get('1', {}).get('precision', 0.0)
    f1_1 = report.get('1', {}).get('f1-score', 0.0)
    accuracy = report.get('accuracy', 0.0)
    bal_acc = (recall0 + recall1) / 2
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    if recall1 >= TARGET_RECALL_POS and recall0 >= MIN_RECALL_NEG:
        tier = 3
    elif recall1 >= FALLBACK_RECALL_POS and recall0 >= MIN_RECALL_NEG:
        tier = 2
    elif recall1 >= TARGET_RECALL_POS:
        tier = 1
    else:
        tier = 0

    # Dentro del objetivo, preferimos mayor specificity; después PPV/balanced accuracy.
    selection_score = (tier, recall0, precision1, bal_acc, accuracy, recall1)

    u1, a1, d1, u2, a2, d2 = params
    return {
        'seed': int(seed),
        'pos_weight_multiplier': float(pos_weight_multiplier),
        'u1': u1, 'a1': a1, 'd1': d1,
        'u2': u2, 'a2': a2, 'd2': d2,
        'threshold': float(THRESHOLD),
        'recall_0': float(recall0),
        'recall_1': float(recall1),
        'specificity': float(recall0),
        'precision_ppv': float(precision1),
        'npv': float(npv),
        'f1_positive': float(f1_1),
        'accuracy': float(accuracy),
        'balanced_accuracy': float(bal_acc),
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'epochs_ran': int(epochs_ran),
        'tier': int(tier),
        'selection_score': selection_score,
    }


class BestEpochByScreening(Callback):
    """Guarda los pesos del mejor epoch según el mismo criterio de selección final."""
    def __init__(self, X_val, y_val, params, seed, pos_weight_multiplier):
        super().__init__()
        self.X_val = X_val.astype('float32')
        self.y_val = y_val.astype(int)
        self.params = params
        self.seed = seed
        self.pos_weight_multiplier = pos_weight_multiplier
        self.best_row = None
        self.best_weights = None
        self.best_probs = None

    def on_epoch_end(self, epoch, logs=None):
        y_prob = self.model.predict(self.X_val, verbose=0).reshape(-1)
        row = metric_row(
            self.y_val, y_prob, self.params, self.seed,
            epochs_ran=epoch + 1,
            pos_weight_multiplier=self.pos_weight_multiplier,
        )
        if self.best_row is None or row['selection_score'] > self.best_row['selection_score']:
            self.best_row = row
            self.best_weights = self.model.get_weights()
            self.best_probs = y_prob.copy()


# ==============================
# CARGA Y PREPARACIÓN DE DATOS
# ==============================
feat_df = read_csv_fallback('./../data/lista_global_vars.csv', './lista_global_vars.csv')
target_df = read_csv_fallback('./../data/target_col.csv', './target_col.csv').fillna(0)

df_merged = feat_df.join(target_df, how='inner')
df_merged = df_merged[
    ~((df_merged['GENERO_BIN_2'] == 1) | (df_merged['ORIENTSEX.BN_3'] == 1))
].drop(columns=['GENERO_BIN_2', 'ORIENTSEX.BN_3']).reset_index(drop=True)

cols_to_drop = [
    'VÍCTIMA', 'VICTIMA_PERPETRADOR', 'POLIVICTIMIZACION', 'POLIPERPETRACION',
    'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'NO.VICT_NO.PERP', 'V.O',
    'P.SUM.TOTAL', 'V.SUM.TOTAL'
]

df_merged_perpetrador = df_merged.drop(columns=cols_to_drop)
df_features = df_merged_perpetrador.drop(columns=['PERPETRADOR'])
y_series = df_merged_perpetrador['PERPETRADOR'].astype(int)

print('Analytical n:', len(df_features))
print('Target counts:', Counter(y_series))

df_features.to_csv(os.path.join(OUTPUT_DIR, 'df_perpetrador_feat.csv'), index=False)
y_series.to_csv(os.path.join(OUTPUT_DIR, 'df_perpretador_target.csv'), index=False)

# ==============================
# SPLIT PRIMERO; SCALER/PCA SOLO TRAIN
# ==============================
all_idx = df_features.index.to_numpy()
train_idx, val_idx = train_test_split(
    all_idx,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_series,
)

with open(os.path.join(OUTPUT_DIR, 'splits_indices.json'), 'w') as f:
    json.dump({'train': train_idx.tolist(), 'val': val_idx.tolist(), 'pca_scope': 'train_only'}, f, indent=2)

X_train_raw = df_features.loc[train_idx].copy()
X_val_raw = df_features.loc[val_idx].copy()
y_train = y_series.loc[train_idx].values.astype(int)
y_val = y_series.loc[val_idx].values.astype(int)

scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

means = pd.Series(X_train_scaled.mean(axis=0), index=X_train_raw.columns, name='mean_train_scaled')
X_train_centered = X_train_scaled - means.values
X_val_centered = X_val_scaled - means.values

pca = PCA()
X_train_pca_full = pca.fit_transform(X_train_centered)
X_val_pca_full = pca.transform(X_val_centered)

df_variance = pca_variance_df(pca)
n_components = select_n_components(df_variance, PCA_THRESHOLD)

X_train = X_train_pca_full[:, :n_components].astype('float32')
X_val = X_val_pca_full[:, :n_components].astype('float32')
pc_cols = [f'PC{i+1}' for i in range(n_components)]

print(f'PCA components selected @ {PCA_THRESHOLD:.2f}:', n_components)
print(f'Train n={len(train_idx)} positives={int(y_train.sum())}; Val n={len(val_idx)} positives={int(y_val.sum())}')

joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler_minmax.pkl'))
joblib.dump(pca, os.path.join(OUTPUT_DIR, 'modelo_pca.pkl'))
means.to_csv(os.path.join(OUTPUT_DIR, 'medias_escalado.csv'), header=True)
df_variance.to_csv(os.path.join(OUTPUT_DIR, 'df_PCA_variance_trainonly.csv'), index=False)
pd.DataFrame(X_train, columns=pc_cols).to_csv(os.path.join(OUTPUT_DIR, 'X_train.csv'), index=False)
pd.DataFrame(X_val, columns=pc_cols).to_csv(os.path.join(OUTPUT_DIR, 'X_val.csv'), index=False)
pd.Series(y_train, name='PERPETRADOR').to_csv(os.path.join(OUTPUT_DIR, 'y_train.csv'), index=False)
pd.Series(y_val, name='PERPETRADOR').to_csv(os.path.join(OUTPUT_DIR, 'y_val.csv'), index=False)
pd.DataFrame(np.vstack([X_train, X_val]), columns=pc_cols).to_csv(
    os.path.join(OUTPUT_DIR, f'df_PCA_{int(PCA_THRESHOLD*100)}_trainonly_stacked.csv'),
    index=False,
)

counts = Counter(y_train)
base_pos_weight = counts[0] / counts[1]
print('Base class_weight positive ratio:', base_pos_weight)
print('Total trainings:', len(CANDIDATES) * len(POS_WEIGHT_MULTIPLIERS) * len(TRAINING_SEEDS))

# ==============================
# BÚSQUEDA REDUCIDA DETERMINISTA
# ==============================
results = []
best = None
best_report = None
best_pred = None
best_probs = None
best_params = None
best_seed = None
best_pos_weight_multiplier = None
best_model_weights = None
best_model_obj = None

stop_now = False

for cand_i, params in enumerate(CANDIDATES, start=1):
    if stop_now:
        break
    for mult in POS_WEIGHT_MULTIPLIERS:
        if stop_now:
            break
        for seed_i, train_seed in enumerate(TRAINING_SEEDS, start=1):
            candidate_seed = int(train_seed + cand_i * 1000 + int(mult * 100))
            class_weight = {0: 1.0, 1: float(base_pos_weight * mult)}
            print(f'\n[cand {cand_i:02d}/{len(CANDIDATES)} mult={mult} seed={candidate_seed}] params={params} class_weight={class_weight}')

            tf.keras.backend.clear_session()
            random.seed(candidate_seed)
            np.random.seed(candidate_seed)
            tf.keras.utils.set_random_seed(candidate_seed)

            train_ds = make_dataset(X_train, y_train, BATCH_SIZE, seed=candidate_seed, training=True)
            val_ds = make_dataset(X_val, y_val, BATCH_SIZE, seed=None, training=False)
            model = build_model(X_train.shape[1], params, candidate_seed)

            best_epoch_cb = BestEpochByScreening(X_val, y_val, params, candidate_seed, mult)
            callbacks = [
                best_epoch_cb,
                EarlyStopping(monitor='val_loss', mode='min', patience=PATIENCE, restore_best_weights=False),
            ]

            history = model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=EPOCHS,
                class_weight=class_weight,
                callbacks=callbacks,
                verbose=0,
                shuffle=False,
            )

            # Restaurar mejor epoch según criterio screening.
            if best_epoch_cb.best_weights is not None:
                model.set_weights(best_epoch_cb.best_weights)
                y_prob = best_epoch_cb.best_probs
                row = best_epoch_cb.best_row.copy()
            else:
                y_prob = model.predict(X_val.astype('float32'), verbose=0).reshape(-1)
                row = metric_row(y_val, y_prob, params, candidate_seed, len(history.history.get('loss', [])), mult)

            row['epochs_fit_total'] = int(len(history.history.get('loss', [])))
            row['base_pos_weight'] = float(base_pos_weight)
            row['effective_pos_weight'] = float(base_pos_weight * mult)
            results.append(row)

            print({k: row[k] for k in ['recall_1', 'recall_0', 'precision_ppv', 'npv', 'balanced_accuracy', 'accuracy', 'TP', 'FP', 'TN', 'FN', 'tier', 'epochs_ran', 'epochs_fit_total', 'effective_pos_weight']})

            if best is None or row['selection_score'] > best['selection_score']:
                best = row
                best_params = params
                best_seed = candidate_seed
                best_pos_weight_multiplier = mult
                best_probs = y_prob.copy()
                best_pred = (best_probs >= THRESHOLD).astype(int)
                best_report = classification_report(y_val, best_pred, output_dict=True, zero_division=0)
                best_model_weights = model.get_weights()
                best_model_obj = model
                model.save(os.path.join(OUTPUT_DIR, 'best_model.h5'))
                print('  -> Nuevo BEST guardado')

            if (EARLY_EXIT_IF_STRONG_TARGET and row['tier'] == 3 and row['recall_0'] >= EARLY_EXIT_MIN_RECALL0):
                print('\n🚀 Strong target alcanzado; saliendo antes para ahorrar tiempo.')
                stop_now = True
                break

# ==============================
# GUARDADO FINAL
# ==============================
if best is None:
    raise RuntimeError('No se entrenó ningún candidato; revisar configuración.')

# Asegurar que el modelo guardado sea el final seleccionado.
if best_model_obj is not None and best_model_weights is not None:
    best_model_obj.set_weights(best_model_weights)
    best_model_obj.save(os.path.join(OUTPUT_DIR, 'best_model.h5'))

# grid results
serializable_results = []
for r in results:
    rr = dict(r)
    rr.pop('selection_score', None)
    serializable_results.append(rr)

df_results = pd.DataFrame(serializable_results)
df_results.to_csv(os.path.join(OUTPUT_DIR, 'gridsearch_results.csv'), index=False)

with open(os.path.join(OUTPUT_DIR, 'best_report.json'), 'w') as f:
    json.dump(best_report, f, indent=2)

# Predicciones final
df_pred = pd.DataFrame({
    'idx_original': val_idx,
    'y_true_perp': y_val.astype(int),
    'y_pred_perp': best_pred.astype(int),
    'y_prob_perp': best_probs.astype(float),
}).sort_values('idx_original').reset_index(drop=True)
df_pred.to_csv(os.path.join(OUTPUT_DIR, 'predictions_with_probs.csv'), index=False)

metrics_final = pd.DataFrame([{k: best[k] for k in [
    'threshold', 'accuracy', 'balanced_accuracy', 'precision_ppv', 'recall_1',
    'specificity', 'npv', 'f1_positive', 'TP', 'FP', 'TN', 'FN'
]}]).rename(columns={'recall_1': 'recall_sensitivity'})
metrics_final['support'] = len(y_val)
metrics_final.to_csv(os.path.join(OUTPUT_DIR, 'metrics_final.csv'), index=False)

config = {
    'random_state': RANDOM_STATE,
    'force_cpu_for_reproducibility': FORCE_CPU_FOR_REPRODUCIBILITY,
    'enable_strict_op_determinism': ENABLE_STRICT_OP_DETERMINISM,
    'threshold': THRESHOLD,
    'pca_threshold': PCA_THRESHOLD,
    'n_components': int(n_components),
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'learning_rate': LEARNING_RATE,
    'target_recall_pos': TARGET_RECALL_POS,
    'min_recall_neg': MIN_RECALL_NEG,
    'fallback_recall_pos': FALLBACK_RECALL_POS,
    'base_pos_weight': float(base_pos_weight),
    'pos_weight_multipliers': POS_WEIGHT_MULTIPLIERS,
    'training_seeds': TRAINING_SEEDS,
    'candidates': [list(c) for c in CANDIDATES],
    'selection_rule': {
        'tier_3': f'recall_1 >= {TARGET_RECALL_POS} and recall_0 >= {MIN_RECALL_NEG}',
        'tier_2': f'recall_1 >= {FALLBACK_RECALL_POS} and recall_0 >= {MIN_RECALL_NEG}',
        'tier_1': f'recall_1 >= {TARGET_RECALL_POS}',
        'score': '(tier, recall_0, precision_ppv, balanced_accuracy, accuracy, recall_1)',
        'threshold_fixed': THRESHOLD,
        'best_epoch_selected_by_same_screening_score': True,
    },
    'best': {k: v for k, v in best.items() if k != 'selection_score'},
    'best_params': {
        'u1': best_params[0], 'a1': best_params[1], 'd1': best_params[2],
        'u2': best_params[3], 'a2': best_params[4], 'd2': best_params[5],
        'seed': int(best_seed),
        'pos_weight_multiplier': float(best_pos_weight_multiplier),
        'effective_pos_weight': float(base_pos_weight * best_pos_weight_multiplier),
    },
}
with open(os.path.join(OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('\n=== BEST SELECTED ===')
print(pd.Series({k: v for k, v in best.items() if k != 'selection_score'}))
print('\n=== FINAL REPORT — PERPETRATOR CSV ===')
print(classification_report(y_val, best_pred, digits=3, zero_division=0))
print('\n=== TOP 10 GRID RESULTS ===')
print(df_results.sort_values(['tier','recall_0','precision_ppv','balanced_accuracy'], ascending=False).head(10).to_string(index=False))
print('\nGuardado en:', OUTPUT_DIR)
print('Archivos principales: predictions_with_probs.csv, best_report.json, gridsearch_results.csv, metrics_final.csv, config.json, best_model.h5')


In [ ]:

# CHECK opcional: ejecutar SOLO después de que haya terminado la celda anterior.
import json
from pprint import pprint
import pandas as pd
from sklearn.metrics import classification_report

pred_path = './content/perpetrator/predictions_with_probs.csv'
data_set = pd.read_csv(pred_path)

print('=== CHECK predictions_with_probs.csv ===')
print(data_set.head())
print(data_set.shape)
print(list(data_set.columns))
print(classification_report(data_set['y_true_perp'], data_set['y_pred_perp'], digits=3, zero_division=0))

print('\n=== METRICS FINAL ===')
print(pd.read_csv('./content/perpetrator/metrics_final.csv').T)

print('\n=== BEST CONFIG ===')
with open('./content/perpetrator/config.json', 'r') as f:
    config = json.load(f)
pprint(config['best_params'])
print('\nBEST METRICS:')
pprint(config['best'])

print('\n=== TOP 10 GRID RESULTS ===')
df_grid = pd.read_csv('./content/perpetrator/gridsearch_results.csv')
print(df_grid.sort_values(['tier','recall_0','precision_ppv','balanced_accuracy'], ascending=False).head(10).to_string(index=False))
